In [ ]:
"""
exp03 Step 01: "Sell At Strength" Exploration (all-time-high exits; EXPLORATION ON 2005-2014 ONLY)

Hypothesis (Nicolas, 2026-10-04): instead of selling on weakness (exp02 stops), sell when SPY is at or near its all-time
high and buy back later, lower. The same identity as exp02 applies: an episode that sells at S and buys back at R ends
with S / R times the shares, so the idea wins only if prices AFTER near-ATH days tend to FALL (not merely rise less).

This notebook repeats the exploration of 2026-10-04 (ath_exit_exploration / ath_exit_exploration_clean, 30 trials in
the legacy trial log) under this experiment's name, on 2005-01-03 -> 2014-12-31 only (training data of every fold; the
data is cut at so.config.EXPLORATION_DATA_CUTOFF_DATE_STR = 2015-03-18, so no forward return reaches the walk-forward period):
    1. Forward returns after near-ATH days vs all days (n = 1 .. 60 sessions), monthly block bootstrap of the difference.
    2. The 15 rules of PROTOCOL.md §4 over the whole exploration period (one continuous window, start invested).

The walk-forward (step 02) uses ALL 15 rules as candidates: none is picked because of its result here.
Caveats: the "all-time high" is the highest high since 2005-01-03 (2005-2006 highs are pseudo-ATHs), and 2005-2014
contains only about 15 separate ATH episodes.
Prerequisite: the raw data only. Outputs: store04_experiments/exp03_ath_exit/step01_exploration_data/.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.express as px
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp03_ath_exit import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp03_ath_exit.rules import get_rule_candidate_list, simulate_rule_dict, get_rule_metric_dict, get_event_forward_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

In [ ]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = config.EXPLORATION_START_DATE_STR, config.EXPLORATION_END_DATE_STR
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tExploration window:\t{exploration_start_str} -> {exploration_end_str}")

In [ ]:
"""
Check 1: forward log return from the 15:59 fill of a near-ATH day to the 15:59 fill n sessions later, compared with the
same quantity for every day. diff < 0 = prices rise LESS than usual after near-ATH days; selling at S and buying back at R
pays only if the event mean itself is NEGATIVE.
"""
# DEFINE THE EVENTS (THE THREE EXIT THRESHOLDS AND THE 52-WEEK HIGH)
ath_dist_arr = daily_pdf["ath_drawdown_pct"].to_numpy(dtype=float)
high250_dist_arr = np.nan_to_num(daily_pdf["high250_dist_pct"].to_numpy(dtype=float), nan=-1.0)
event_dict = {**{f"ATH within {ath_within:.1%}": ath_dist_arr >= -ath_within for ath_within in exp_config.ATH_WITHIN_LIST},
              f"52-week high within {exp_config.HIGH250_WITHIN:.0%}": high250_dist_arr >= -exp_config.HIGH250_WITHIN}
# CALL FUNCTION TO BUILD THE FORWARD RETURN TABLE
forward_pdf = get_event_forward_pdf(daily_pdf, event_dict, exploration_end_str)
# DISPLAY THE NEAR-ATH DAYS PER YEAR
in_window_arr = np.array([date_object <= pd.Timestamp(exploration_end_str).date() for date_object in daily_pdf["date"]])
print("Days within 1% of the ATH per year:", pd.Series([d.year for d in daily_pdf["date"].to_numpy()[in_window_arr & (ath_dist_arr >= -0.01)]]).value_counts().sort_index().to_dict())
# PREVIEW DATAFRAME
forward_pdf.round(4)

In [ ]:
"""
Check 2: the 15 rules (one continuous window, start invested, same costs as exp02, forced buy-back after 60 decisions).
One of them doing well while its neighbours do badly is what noise looks like.
"""
# SIMULATE BUY AND HOLD
buy_hold_metric_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)["metric_dict"]
# LISTS TO HOLD THE RESULTS
rule_dict_list, simulation_dict_dict = [], {}
# ITERATE OVER THE RULES
for candidate_dict in get_rule_candidate_list():
    # SIMULATE THE RULE
    simulation_dict = simulate_rule_dict(ohlcv_array_dict, daily_pdf, exploration_start_str, exploration_end_str, candidate_dict)
    simulation_dict_dict[(candidate_dict["ath_within"], candidate_dict["rule"])] = simulation_dict
    # STORE THE METRICS
    rule_dict_list.append({"ath_within": candidate_dict["ath_within"], "rule": candidate_dict["rule"], **get_rule_metric_dict(simulation_dict, buy_hold_metric_dict["total_return"])})
# CONVERT TO A DATAFRAME
rule_pdf = pd.DataFrame(rule_dict_list)
# DISPLAY INFORMATION
print(f"Buy & hold:\ttotal return {buy_hold_metric_dict['total_return']:.2%} | Sharpe {buy_hold_metric_dict['sharpe_ratio']:.2f} | max drawdown {buy_hold_metric_dict['max_drawdown']:.2%}")
# DISPLAY THE EXCESS RETURN GRID (THRESHOLD x BUY-BACK)
display(rule_pdf.pivot(index="ath_within", columns="rule", values="excess_return").round(4))
# PREVIEW DATAFRAME
rule_pdf.round(4)

In [ ]:
# PREVIEW THE EPISODES OF ONE RULE (WITHIN 1% OF THE ATH, BUY BACK 5% LOWER OR AFTER 60 DECISIONS)
simulation_dict_dict[(0.01, "dip_5pct")]["episode_pdf"][["exit_ts", "exit_fill_price", "reentry_ts", "reentry_fill_price", "cash_session_count", "reentry_reason", "share_gain_pct"]]

In [ ]:
# SAVE THE RESULTS
write_csv_file_to_path(forward_pdf, f"{output_path_str}forward_return_data.csv", "W")
write_csv_file_to_path(rule_pdf, f"{output_path_str}rule_data.csv", "W")
# LOG ONE EXPLORATION ENTRY PER RULE (NO VALIDATION OR TEST WINDOW IS TOUCHED)
logged_pdf = log_trial_pdf(exp_config, "exploration", rule_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["ath_within", "rule", "window"],
                           ["total_return", "excess_return", "sharpe_ratio", "max_drawdown", "exit_count", "mean_log_share_gain"], False,
                           note_str_in="exp03 step01 sell-at-strength exploration (2005-2014 only)")
print(f"Logged {len(logged_pdf)} exploration entries (config hash {get_config_hash_str(exp_config)})")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)